# Public portfolio version - Gold layer

This notebook is a sanitized public portfolio version focused on the Gold layer of the SenseTemp project. It shows alert classification, reporting, and visualization outputs built on curated Silver data.

# Gold layer overview

The Gold layer exposes business-ready analytical outputs. It applies alert logic, summarizes the thermal behavior of each sensor, and prepares results for reporting and visualization.

## What happens here

* classify each reading as `OK`, `BAJA`, or `ALTA`
* calculate deviation from configured thresholds
* build a Gold summary report by sensor
* expose final tables and charts for portfolio presentation

## Why this matters

This layer is the closest to business consumption. It is where engineering work becomes operational insight.

In [0]:
# Public portfolio configuration example
storage_account = "<your-storage-account>"

path_telemetria_validada = f"wasbs://silver@{storage_account}.blob.core.windows.net/telemetria_validada_delta/"
path_metadata_delta = f"wasbs://silver@{storage_account}.blob.core.windows.net/metadata_sensores_delta/"
path_reporte_detallado = f"wasbs://gold@{storage_account}.blob.core.windows.net/reportes_temperatura_detallado_delta/"

print("Gold portfolio notebook loaded.")
print("Replace demo paths and configure secure access before execution.")

In [0]:
# Read curated Silver outputs required for Gold processing
from pyspark.sql.functions import expr

df_telemetria_clean = spark.read.format("delta").load(path_telemetria_validada)
df_metadata_delta = spark.read.format("delta").load(path_metadata_delta)

sensor_cols = [c for c in df_telemetria_clean.columns if c.startswith("Temp_")]
if not sensor_cols:
    raise ValueError("No valid sensors with metadata are available for Gold processing.")

stack_expr = ", ".join([f"'{c}', `{c}`" for c in sensor_cols])

df_telemetria_long = df_telemetria_clean.select(
    "FechaHoraRecepcion",
    "NombreDispositivo",
    expr(f"stack({len(sensor_cols)}, {stack_expr}) as (SensorID, Temperatura)"),
)

df_joined = df_telemetria_long.join(df_metadata_delta, on="SensorID", how="inner")
display(df_joined)

In [0]:
# Classify alerts and calculate deviation
from pyspark.sql.functions import col, lit, when

df_alertas = (
    df_joined
    .withColumn(
        "EstadoAlerta",
        when(col("Temperatura") < col("TempMin"), lit("BAJA"))
        .when(col("Temperatura") > col("TempMax"), lit("ALTA"))
        .otherwise(lit("OK")),
    )
    .withColumn(
        "Desvio",
        when(col("Temperatura") < col("TempMin"), col("Temperatura") - col("TempMin"))
        .when(col("Temperatura") > col("TempMax"), col("Temperatura") - col("TempMax"))
        .otherwise(lit(0.0)),
    )
)

display(df_alertas.orderBy("FechaHoraRecepcion", "SensorID"))
display(df_alertas.groupBy("EstadoAlerta").count().orderBy("EstadoAlerta"))

In [0]:
# Build Gold summary report
from pyspark.sql.functions import (
    avg,
    count,
    min as min_,
    max as max_,
    sum as sum_,
    when,
    col,
    to_timestamp,
    unix_timestamp,
    current_timestamp,
    lit,
)

df_alertas_ts = df_alertas.withColumn("FechaHoraTs", to_timestamp("FechaHoraRecepcion"))

reporte_sensor = (
    df_alertas_ts
    .groupBy("SensorID", "Ubicacion")
    .agg(
        count("*").alias("TotalLecturas"),
        sum_(when(col("EstadoAlerta") == "OK", 1).otherwise(0)).alias("LecturasEnRango"),
        sum_(when(col("EstadoAlerta") == "BAJA", 1).otherwise(0)).alias("LecturasBajas"),
        sum_(when(col("EstadoAlerta") == "ALTA", 1).otherwise(0)).alias("LecturasAltas"),
        avg("Temperatura").alias("TempPromedio"),
        min_("Temperatura").alias("TempMinObservada"),
        max_("Temperatura").alias("TempMaxObservada"),
        min_("TempMin").alias("LimiteMin"),
        max_("TempMax").alias("LimiteMax"),
        min_("FechaHoraTs").alias("PrimeraLectura"),
        max_("FechaHoraTs").alias("UltimaLectura"),
    )
)

df_fuera_rango = df_alertas_ts.filter(col("EstadoAlerta") != "OK")

if df_fuera_rango.count() > 0:
    duracion_alertas = (
        df_fuera_rango
        .groupBy("SensorID", "EstadoAlerta")
        .agg(
            count("*").alias("CantidadEventos"),
            min_("FechaHoraTs").alias("PrimeraVezFueraRango"),
            max_("FechaHoraTs").alias("UltimaVezFueraRango"),
            avg("Desvio").alias("DesvioPromedio"),
            min_("Desvio").alias("DesvioMinimo"),
            max_("Desvio").alias("DesvioMaximo"),
        )
        .withColumn("DuracionSegundos", unix_timestamp("UltimaVezFueraRango") - unix_timestamp("PrimeraVezFueraRango"))
        .withColumn("DuracionMinutos", col("DuracionSegundos") / 60.0)
    )

    reporte_completo = (
        reporte_sensor
        .join(duracion_alertas, on="SensorID", how="left")
        .withColumn("TieneAlertas", when(col("EstadoAlerta").isNotNull(), lit("SI")).otherwise(lit("NO")))
        .withColumn("FechaProceso", current_timestamp())
        .orderBy("SensorID", "EstadoAlerta")
    )
else:
    reporte_completo = (
        reporte_sensor
        .withColumn("TieneAlertas", lit("NO"))
        .withColumn("EstadoAlerta", lit(None).cast("string"))
        .withColumn("CantidadEventos", lit(None).cast("integer"))
        .withColumn("PrimeraVezFueraRango", lit(None).cast("timestamp"))
        .withColumn("UltimaVezFueraRango", lit(None).cast("timestamp"))
        .withColumn("DuracionSegundos", lit(None).cast("double"))
        .withColumn("DuracionMinutos", lit(None).cast("double"))
        .withColumn("DesvioPromedio", lit(None).cast("double"))
        .withColumn("DesvioMinimo", lit(None).cast("double"))
        .withColumn("DesvioMaximo", lit(None).cast("double"))
        .withColumn("FechaProceso", current_timestamp())
        .orderBy("SensorID")
    )

(
    reporte_completo.write.format("delta").mode("overwrite").save(path_reporte_detallado)
)

print(f"Gold report written to demo path: {path_reporte_detallado}")
print(f"Total monitored sensors: {reporte_sensor.count()}")
if df_fuera_rango.count() > 0:
    print(f"Sensors with out-of-range alerts: {duracion_alertas.select('SensorID').distinct().count()}")
else:
    print("No out-of-range alerts detected")

display(reporte_completo)

In [0]:
# Visualize temperature trends by sensor
from pyspark.sql.functions import to_timestamp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

series_sensor = (
    df_alertas
    .withColumn("FechaHoraTs", to_timestamp("FechaHoraRecepcion"))
    .select(
        "SensorID",
        "FechaHoraRecepcion",
        "FechaHoraTs",
        "Temperatura",
        "TempMin",
        "TempMax",
        "EstadoAlerta",
    )
    .orderBy("SensorID", "FechaHoraTs")
)

display(series_sensor)

pdf_series = series_sensor.toPandas().sort_values(["SensorID", "FechaHoraTs"])
sensores = sorted(pdf_series["SensorID"].unique())

fig, axes = plt.subplots(len(sensores), 1, figsize=(14, 3.5 * len(sensores)), sharex=True)
if len(sensores) == 1:
    axes = [axes]

for ax, sensor in zip(axes, sensores):
    data_sensor = pdf_series[pdf_series["SensorID"] == sensor].copy()

    ax.plot(data_sensor["FechaHoraTs"], data_sensor["Temperatura"], marker="o", linewidth=2, label="Temperatura")
    ax.axhline(data_sensor["TempMin"].iloc[0], color="red", linestyle="--", label="TempMin")
    ax.axhline(data_sensor["TempMax"].iloc[0], color="orange", linestyle="--", label="TempMax")

    fuera_rango = data_sensor[data_sensor["EstadoAlerta"] != "OK"]
    if not fuera_rango.empty:
        ax.scatter(fuera_rango["FechaHoraTs"], fuera_rango["Temperatura"], color="crimson", s=60, label="Fuera de rango")

    ax.set_title(sensor)
    ax.set_ylabel("Temperatura")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# What to show in GitHub

For a public repository, this Gold notebook should demonstrate:

* alert classification logic
* final Gold reporting output
* summary tables suitable for screenshots
* trend visualizations for the README

Recommended GitHub file name:

`notebooks/03_gold_reporting_visualization.ipynb`